In [3]:
import pandas as pd
import folium
from folium.plugins import HeatMap
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
import pandas as pd

# 1. Vos données (Exemple)
df = pd.read_csv("polytech2025edd_tp_source\\tb_region_data.csv")

# 2. Initialiser le géocodeur (Nominatim est un service gratuit d'OpenStreetMap)
geolocator = Nominatim(user_agent="mon_application_heatmap_ventes")
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)

# Fonction pour obtenir les coordonnées
def get_lat_lon(city, country):
    try:
        # On ajoute ", France" pour être plus précis (optionnel)
        location = geocode(city + country)
        return location.latitude, location.longitude
    except:
        return None, None

print("Récupération des coordonnées GPS en cours...")

# Appliquer la fonction à chaque ville
# Note : Cela peut prendre du temps si vous avez beaucoup de villes
df['Coordinates'] = df.apply(lambda row: get_lat_lon(row['sales_city'], row['sales_country']), axis=1)

# Séparer lat et lon et nettoyer les données (supprimer les villes non trouvées)
df[['Lat', 'Lon']] = pd.DataFrame(df['Coordinates'].tolist(), index=df.index)
df_clean = df.dropna(subset=['Lat', 'Lon'])

df_clean:

Récupération des coordonnées GPS en cours...


,region_id,sales_city,sales_state_province,sales_district,sales_region,sales_country,sales_district_id,Coordinates,Lat,Lon


In [ ]:
# 3. Création de la Carte de Chaleur
# On centre la carte sur la moyenne des coordonnées
map_center = [df_clean['Lat'].mean(), df_clean['Lon'].mean()]
m = folium.Map(location=map_center, zoom_start=6)

# Préparer les données pour le plugin HeatMap : liste de [Lat, Lon, Poids]
heat_data = df_clean[['Lat', 'Lon', 'Ventes']].values.tolist()

# Ajouter la couche HeatMap
HeatMap(heat_data, radius=25, blur=15, max_zoom=1).add_to(m)

# 4. Sauvegarder la carte
output_file = "carte_chaleur_ventes.html"
m.save(output_file)

print(f"Carte générée avec succès : {output_file}")